### **Multi-Agent Conversation and Stand-up Comedy**

In [26]:
from dotenv import load_dotenv, find_dotenv
import os, openai
from pprint import pprint

_ = load_dotenv(find_dotenv())
openai.api_key = os.environ['OPENAI_API_KEY']

llm_config = {'model': 'gpt-3.5-turbo'}

#### **Define an AutoGen agent**

In [27]:
#!pip install pyautogen

from autogen import ConversableAgent

agent = ConversableAgent(
    name = 'chatbot',
    llm_config = llm_config,
    human_input_mode = 'NEVER',
)

reply = agent.generate_reply(
    messages=[{'content': 'Tell me a joke.', 'role': 'user'}]
)
print(reply)

Sure, here's a joke for you:

Why did the scarecrow win an award?
Because he was outstanding in his field!


In [28]:
reply = agent.generate_reply(
    messages=[{'content': 'Repeat the joke.', 'role': 'user'}]
)
print(reply)

I'd be happy to! Which joke would you like to hear again?


#### **Conversation**

In [29]:
cathy = ConversableAgent(
    name = 'cathy',
    system_message = 'Your name is Cathy and you are a stand-up comedian.',
    llm_config = llm_config,
    human_input_mode = 'NEVER',
)

joe = ConversableAgent(
    name = 'joe',
    system_message = 'Your name is Joe and you are a stand-up comedian. '
                     'Start the next joke from the punchline of the previous joke.',
    llm_config = llm_config,
    human_input_mode = 'NEVER',
)

chat = joe.initiate_chat(
    recipient = cathy, 
    message = 'I\'m Joe. Cathy, let\'s keep the jokes rolling.',
    max_turns = 2,
)

joe (to cathy):

I'm Joe. Cathy, let's keep the jokes rolling.

--------------------------------------------------------------------------------
cathy (to joe):

Hey Joe, ready for some laughs? Alright, here we go! Why did the golfer bring two pairs of pants? In case he got a hole in one!

--------------------------------------------------------------------------------
joe (to cathy):

Nice one, Cathy! Speaking of sports, did you hear about the football player who went to the bank?

--------------------------------------------------------------------------------
cathy (to joe):

I'm not sure, what happened when the football player went to the bank?

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (17e5c860-3fa2-4251-a3bb-5b6aaff13b5b): Maximum turns (2) reached


In [30]:
pprint(chat.chat_history)

[{'content': "I'm Joe. Cathy, let's keep the jokes rolling.",
  'name': 'joe',
  'role': 'assistant'},
 {'content': 'Hey Joe, ready for some laughs? Alright, here we go! Why did the '
             'golfer bring two pairs of pants? In case he got a hole in one!',
  'name': 'cathy',
  'role': 'user'},
 {'content': 'Nice one, Cathy! Speaking of sports, did you hear about the '
             'football player who went to the bank?',
  'name': 'joe',
  'role': 'assistant'},
 {'content': "I'm not sure, what happened when the football player went to the "
             'bank?',
  'name': 'cathy',
  'role': 'user'}]


In [31]:
pprint(chat.cost)

{'usage_excluding_cached_inference': {'gpt-3.5-turbo-0125': {'completion_tokens': 72,
                                                             'cost': 0.0002255,
                                                             'prompt_tokens': 235,
                                                             'total_tokens': 307},
                                      'total_cost': 0.0002255},
 'usage_including_cached_inference': {'gpt-3.5-turbo-0125': {'completion_tokens': 72,
                                                             'cost': 0.0002255,
                                                             'prompt_tokens': 235,
                                                             'total_tokens': 307},
                                      'total_cost': 0.0002255}}


In [32]:
pprint(chat.summary)

"I'm not sure, what happened when the football player went to the bank?"


#### **Get a better summary**

In [35]:
chat = joe.initiate_chat(
    cathy, 
    message = 'I\'m Joe. Cathy, let\'s keep the jokes rolling.',
    max_turns = 2, 
    summary_method = 'reflection_with_llm',
    summary_prompt = 'Summarize the conversation'
)

joe (to cathy):

I'm Joe. Cathy, let's keep the jokes rolling.

--------------------------------------------------------------------------------
cathy (to joe):

Hey Joe! Glad to see you're ready for some laughs. Let's jump right in! So, I recently went on a diet where I cut out all carbs. It was tough, but I stuck with it. Now, I can proudly say that I still don't have abs, but I do have a great relationship with my refrigerator.

--------------------------------------------------------------------------------
joe (to cathy):

That's impressive, Cathy! It's all about that strong bond with your fridge. Speaking of food, I recently tried to become a vegetarian, but it didn't last long. I just couldn't find the soul in my tofu.

--------------------------------------------------------------------------------
cathy (to joe):

I hear you, Joe! Tofu can be a tough sell sometimes. It's like the rebellious teenager of the food world—it just won't conform to your expectations. But hey, at leas

In [36]:
pprint(chat.summary)

('Joe and Cathy shared jokes about their experiences with dieting and '
 'vegetarianism, highlighting the struggles and humor that come with trying '
 'new eating habits.')


#### **Chat Termination**

In [37]:
cathy = ConversableAgent(
    name = 'cathy',
    system_message =
    'Your name is Cathy and you are a stand-up comedian. '
    "When you're ready to end the conversation, say 'I gotta go'.",
    llm_config = llm_config,
    human_input_mode = 'NEVER',
    is_termination_msg = lambda msg: 'I gotta go' in msg['content'],
)

joe = ConversableAgent(
    name = 'joe',
    system_message =
    'Your name is Joe and you are a stand-up comedian. '
    "When you're ready to end the conversation, say 'I gotta go'.",
    llm_config = llm_config,
    human_input_mode = 'NEVER',
    is_termination_msg = lambda msg: 'I gotta go' in msg['content'] or 'Goodbye' in msg['content'],
)

chat_result = joe.initiate_chat(
    recipient = cathy,
    message = "I'm Joe. Cathy, let's keep the jokes rolling."
)

joe (to cathy):

I'm Joe. Cathy, let's keep the jokes rolling.

--------------------------------------------------------------------------------
cathy (to joe):

Hey Joe! Thanks for tuning in for some laughs. So, did you hear about the mathematician who's afraid of negative numbers? He'll stop at nothing to avoid them!

--------------------------------------------------------------------------------
joe (to cathy):

Haha, that's a good one, Cathy! I guess he's positively terrified of negative numbers! So, what's your favorite type of joke to tell?

--------------------------------------------------------------------------------
cathy (to joe):

I gotta go

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (8735c22e-7422-4b60-9687-392c874e2ee6): Termination message condition on agent 'joe' met


In [38]:
cathy.send(message = "What's last joke we talked about?", recipient = joe)

cathy (to joe):

What's last joke we talked about?

--------------------------------------------------------------------------------
joe (to cathy):

The joke we shared was about the mathematician who's afraid of negative numbers: He'll stop at nothing to avoid them!

--------------------------------------------------------------------------------
cathy (to joe):

Oh right, thanks! So back to your question, I love all types of jokes, but I particularly enjoy puns and wordplay. There's just something about clever word combinations that never fails to tickle my funny bone. How about you, Joe? Do you have a favorite type of joke?

--------------------------------------------------------------------------------
joe (to cathy):

I gotta go

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (83be4320-d5cd-4a3a-bd23-b555dfd301d0): Termination message condition on agent 'cathy' met
